In [2]:
import torch

text = ('python models learn from examples. students inspect errors and improve models. ') * 30

chars = sorted(set(text))
to_id = {c: i for i, c in enumerate(chars)}
to_char = {i: c for c, i in to_id.items()}

encoded = torch.tensor([to_id[c] for c in text], dtype=torch.long)

print(len(text), len(chars), chars)

2370 21 [' ', '.', 'a', 'c', 'd', 'e', 'f', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'x', 'y']


This code turns characters into numbers so the model can read them. If a character is missing, the model won't know what it is or how to use it.

In [3]:
n = len(encoded)

train_ids = encoded[:int(.7*n)]
val_ids = encoded[int(.7*n):int(.85*n)]
test_ids = encoded[int(.85*n):]

window = 20

def pairs(ids):
    X = torch.stack([ids[i:i+window] for i in range(len(ids)-window)])
    y = ids[window:]
    return X, y

X_train, y_train = pairs(train_ids)
X_val, y_val = pairs(val_ids)
X_test, y_test = pairs(test_ids)

print(X_train.shape, X_val.shape, X_test.shape)

torch.Size([1639, 20]) torch.Size([335, 20]) torch.Size([336, 20])


This splits the data into training, validation, and testing sets. Because the same sentences repeat, the model can memorize patterns instead of learning something new.

In [4]:
from torch import nn

class CharGRU(nn.Module):
    def __init__(self, vocab):
        super().__init__()
        self.embed = nn.Embedding(vocab, 8)
        self.gru = nn.GRU(8, 24, batch_first=True)
        self.head = nn.Linear(24, vocab)

    def forward(self, x):
        out, _ = self.gru(self.embed(x))
        return self.head(out[:, -1, :])

torch.manual_seed(42)
model = CharGRU(len(chars))

print(model(X_train[:4]).shape)

torch.Size([4, 21])


This creates a model that looks at characters and tries to predict the next one. The model gives each character a score so it can pick the most likely one.

In [5]:
from torch.utils.data import TensorDataset, DataLoader
from collections import Counter

majority = Counter(y_train.tolist()).most_common(1)[0][0]

print('majority validation accuracy:', (y_val == majority).float().mean().item())

loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)

opt = torch.optim.Adam(model.parameters(), lr=.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(15):
    model.train()
    for xb, yb in loader:
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()

model.eval()

with torch.inference_mode():
    val_pred = model(X_val).argmax(1)

print('validation accuracy:', (val_pred == y_val).float().mean().item())

majority validation accuracy: 0.13731342554092407
validation accuracy: 1.0


This trains the model and checks how accurate its predictions are.  Because the model can predict characters correctly without understanding what the words mean.

In [6]:
prompt = text[:window]

input_ids = torch.tensor([[to_id[c] for c in prompt]])

with torch.inference_mode():
    next_id = model(input_ids).argmax(1).item()

print(repr(prompt), '->', repr(to_char[next_id]))

'python models learn ' -> 'f'


This uses the model to predict the next character based on the text. It would help the model learn more patterns and make better predictions.

In [7]:
with torch.inference_mode():
    test_pred = model(X_test).argmax(1)

print('final test accuracy:', (test_pred == y_test).float().mean().item())

final test accuracy: 1.0


This checks how accurate the model is on the test data. Predicting characters is just finding patterns. Understanding meaning requires knowing what the words actually mean.

My model got 100% validation accuracy, which is a lot better than the 13.73% baseline. The model learned the patterns well, but the text was repetitive, making it easier to predict. Next time, I would use different sentences to see how well the model performs on new text.

